
# 34 组合优化：均值方差、风险平价与 Black–Litterman

> **本章要解决的问题**：给定预期收益和协方差，权重怎么算？为什么均值方差的最优权重经常“极端又不可执行”？风险平价和 Black–Litterman 各自修正了什么？

## 本章导读

第07章讲优化工具，第30章讲期权组合风险。这一章讲**组合构建的三种主流方法**：

1. 均值方差：Markowitz 有效前沿；
2. 风险平价：按风险贡献分配；
3. Black–Litterman：市场均衡 + 主观观点。

再加两件实务：估计误差/收缩、约束/换手/稳健。

### 学习目标

- 能写均值方差问题，并算最小方差/切点组合。
- 能画/读有效前沿，解释约束对前沿的影响。
- 能用收缩/重抽样降低权重对 $\mu$ 的敏感。
- 能算风险平价权重和风险贡献。
- 能用 Black–Litterman 把观点并入均衡收益。
- 能把换手/上限/整数约束加进优化。

### 先修

第07章的 QP/拉格朗日；第04章的协方差/特征值；会 scipy。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="34 组合优化：均值方差、风险平价与 BL" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>34 组合优化：均值方差、风险平价与 BL｜知识图</title>
<desc>本章主线：均值方差的前沿很漂亮但权重极端；收缩、风险平价与 BL 观点把它拉回可执行。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a34" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao34" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag34" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">34</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">组合优化：均值方差、风险平价与 BL</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：均值方差的前沿很漂亮但权重极端；收缩、风险平价与 BL 观点把它拉回可执行。</text>
<path d="M 80 330 L 555 330" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag34)"/>
<path d="M 80 330 L 80 330" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag34)"/>
<path d="M 80 330 L 80 330" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag34)"/>
<path d="M 80 330 L 80 110" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag34)"/>
<text x="540" y="348" font-size="9.5" fill="#6E7B8B">风险 σ</text>
<text x="88" y="122" font-size="9.5" fill="#6E7B8B">收益 μ</text>
<path d="M 175 298 L 215 262 L 265 228 L 325 202 L 390 186 L 455 176 L 520 170" fill="none" stroke="#1B3A5C" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<text x="300" y="168" font-size="9.6" fill="#1B3A5C" font-weight="bold">有效前沿</text>
<circle cx="140" cy="302" r="3.6" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.2"/>
<text x="146" y="298" font-size="8.4" fill="#6E7B8B">A</text>
<circle cx="160" cy="288" r="3.6" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.2"/>
<text x="166" y="284" font-size="8.4" fill="#6E7B8B">B</text>
<circle cx="190" cy="296" r="3.6" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.2"/>
<text x="196" y="292" font-size="8.4" fill="#6E7B8B">C</text>
<circle cx="225" cy="282" r="3.6" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.2"/>
<text x="231" y="278" font-size="8.4" fill="#6E7B8B">D</text>
<circle cx="175" cy="298" r="5" fill="#FFFFFF" stroke="#E07B39" stroke-width="1.8"/>
<text x="120" y="318" font-size="8.6" fill="#E07B39">最小方差</text>
<line x1="90" y1="330" x2="470" y2="148" stroke="#E07B39" stroke-width="1.6" stroke-linecap="round" stroke-dasharray="6 4"/>
<circle cx="390" cy="186" r="5.2" fill="#E07B39"/>
<text x="398" y="206" font-size="8.8" fill="#E07B39">CML / 切点</text>
<circle cx="455" cy="176" r="4.5" fill="#FFFFFF" stroke="#1B3A5C" stroke-width="1.7"/>
<text x="463" y="196" font-size="8.6" fill="#1B3A5C">风险平价</text>
<path d="M 185 290 L 235 252 L 300 216 L 370 194 L 445 182 L 515 174" fill="none" stroke="#6E7B8B" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="6 4"/>
<text x="445" y="160" font-size="8.8" fill="#6E7B8B">BL 调整后前沿</text>
<rect x="600" y="100" width="155" height="60" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="677.5" y="122.4" font-size="10.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">均值方差 MVO</text>
<text x="677.5" y="140.2" font-size="8.4" fill="#6E7B8B" text-anchor="middle">前沿漂亮，权重极端</text>
<rect x="600" y="182" width="155" height="60" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="677.5" y="204.4" font-size="10.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">风险平价</text>
<text x="677.5" y="222.2" font-size="8.4" fill="#6E7B8B" text-anchor="middle">风险贡献相等</text>
<rect x="600" y="264" width="155" height="60" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="677.5" y="286.4" font-size="10.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">Black–Litterman</text>
<text x="677.5" y="304.2" font-size="8.4" fill="#6E7B8B" text-anchor="middle">均衡收益 + 主观观点</text>
<path d="M 598 130 L 522 171" fill="none" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="3 3" marker-start="url(#ag34)"/>
<path d="M 598 214 L 458 179" fill="none" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="3 3" marker-start="url(#ag34)"/>
<path d="M 598 296 L 518 176" fill="none" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="3 3" marker-start="url(#ag34)"/>
<text x="80" y="378" font-size="9.2" fill="#6E7B8B">收缩估计：Σ 加噪、μ 收缩；约束：换手、上限、整数手数</text>
</svg>

> **图**：本章主线：均值方差的前沿很漂亮但权重极端；收缩、风险平价与 BL 观点把它拉回可执行。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：组合研究员、资管/FOF、风险预算、因子 PM。
- **输入**：预期收益、协方差、基准/观点、约束、成本。
- **输出**：权重、有效前沿、风险贡献、换手/可执行方案。
- **在流程的哪一步**：从“预测”到“组合”的构建。
- **失败模式**：把 $\mu$ 当已知、无约束极端权重、忽略换手/容量。
- **最小工作流**：估 $\mu/\Sigma$ → 收缩 → 选方法 → 约束 → 求解 → 风险/换手复查。

---

## 1. 均值方差：有效前沿

### 1.1 问题

$$\min_w\ \frac12 w^T\Sigma w\quad\text{s.t.}\quad w^T\mu\ge r_{\min},\ \mathbf1^Tw=1,\ 0\le w\le1.$$

**公式拆解**
- **符号**：$w$ 是 $N$ 只资产的权重向量（和为 1、无单位），$\Sigma$ 是收益协方差矩阵（收益率平方），$\mu$ 是预期收益向量，$r_{\min}$ 是目标最低收益，$\mu$ 与 $\Sigma$ 只能用调仓日之前的数据估计。
- **结构**：目标 $\frac12w^T\Sigma w$ 是组合方差、最小化它等于在给定收益下把风险压到最低，约束分别是收益门槛、资金守恒与不许卖空，把 $r_{\min}$ 从低到高扫一遍就得到有效前沿。
- **代入**：$\mu=(8\%,3\%)$、$\sigma=(20\%,10\%)$、$\rho=0.2$、$r_{\min}=5\%$ 时取 $w=(0.4,0.6)$ 恰好满足门槛，组合方差 0.01192、波动约 10.92%。
- **边界**：$r_{\min}$ 高于最大单资产收益（8%）时可行域为空、模型无解，$r_{\min}$ 很低时收益约束不绑定、退化为最小方差组合。
- **用途**：作为有效前沿的理论基准检查别的组合是否在前沿之下；不能把样本 $\mu$ 当已知，否则 $\Sigma^{-1}$ 会把估计误差放大成极端权重。

### 1.2 两个特解

最小方差：

$$w_{\min}=\frac{\Sigma^{-1}\mathbf1}{\mathbf1^T\Sigma^{-1}\mathbf1}.$$

切点（最大 Sharpe，$r_f=0$）：

$$w_{\tan}=\frac{\Sigma^{-1}\mu}{\mathbf1^T\Sigma^{-1}\mu}.$$

**公式拆解**
- **符号**：$w_{\tan}$ 是最大 Sharpe 组合的权重，$\mu$ 通常取超额收益，$\Sigma$ 是收益协方差，分母 $\mathbf1^T\Sigma^{-1}\mu$ 只是保证权重和为 1 的归一化常数。
- **结构**：分子 $\Sigma^{-1}\mu$ 是用协方差白化后的收益信号，除以总和后得到权重，它正是所有有效组合中 Sharpe 最高的那一个，也与特定风险厌恶下的均值方差解重合。
- **代入**：沿用 $\mu=(8\%,3\%)$、$r_f=2\%$ 与上面的 $\Sigma$，算出 $\Sigma^{-1}(\mu-r_f)=(1.458,0.417)$，归一化得 $w_{\tan}=(0.778,0.222)$，Sharpe 约 0.303。
- **边界**：超额收益全为 0 时切点无定义、最优是持有现金，$\Sigma$ 奇异时不可解，禁止卖空后必须带约束重新求解。
- **用途**：给出理论最大 Sharpe 的可计算基准，用于检验组合是否接近前沿；不能直接实盘，它样本内集中、带卖空且对 $\mu$ 的误差极其敏感。

### 1.3 代码：前沿



In [2]:

import numpy as np
from scipy.optimize import minimize

mu = np.array([0.06, 0.08, 0.12])
vol = np.array([0.10, 0.15, 0.25])
corr = np.array([[1.0, 0.2, 0.3], [0.2, 1.0, 0.5], [0.3, 0.5, 1.0]])
cov = np.outer(vol, vol) * corr
one = np.ones(3)

w_min = minimize(
    lambda w: w @ cov @ w,
    x0=one / 3,
    bounds=[(0, 1)] * 3,
    constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
    method="SLSQP",
).x
w_tan = np.linalg.solve(cov, mu) / (one @ np.linalg.solve(cov, mu))
for name, w in [("最小方差", w_min), ("切点", w_tan)]:
    print(name, np.round(w, 3), "收益", f"{w @ mu:.2%}", "波动", f"{np.sqrt(w @ cov @ w):.2%}")


最小方差 [0.736 0.264 0.   ] 收益 6.53% 波动 9.03%
切点 [0.615 0.306 0.08 ] 收益 7.09% 波动 9.51%



**读数**：最小方差权重偏债券/低波动资产；切点权重偏高收益资产，可能很集中。**理论最优的“集中”是估计误差的放大器。**

**常见错误**：

1. 不约束卖空，权重出现 −3。
2. 用原始 $\mu$ 不收缩。
3. 把切点当“最优”。

---

## 2. 估计误差：为什么权重会极端

### 2.1 $\mu$ 的误差

$\mu$ 的估计误差远大于 $\Sigma$；$\Sigma^{-1}\mu$ 会把误差放大。样本越短，权重越极端。

### 2.2 收缩

把协方差向对角/单位矩阵拉：

$$\Sigma_{\text{shrunk}}=(1-\lambda)\Sigma+\lambda F.$$

对 $\mu$ 也可以用贝叶斯收缩/置信区间下界。

### 2.3 代码：收缩前后



In [3]:

lam = 0.5
target = np.eye(3) * np.trace(cov) / 3
cov_s = (1 - lam) * cov + lam * target
w_raw = np.linalg.solve(cov, mu)
w_shr = np.linalg.solve(cov_s, mu)
print("原始权重", np.round(w_raw, 3), "条件数", round(np.linalg.cond(cov), 2))
print("收缩权重", np.round(w_shr, 3), "条件数", round(np.linalg.cond(cov_s), 2))
print("权重绝对变化", round(np.abs(w_raw - w_shr).sum(), 3))


原始权重 [4.814 2.393 0.624] 条件数 7.95
收缩权重 [2.377 2.154 1.93 ] 条件数 2.53
权重绝对变化 3.982



**读数**：收缩把极端权重拉回，条件数下降；代价是偏离“理论最优”。**收缩是用偏差换方差。**

**常见错误**：

1. 用全样本估 $\mu$ 再回测同一段。
2. 只收缩 $\Sigma$，不收缩 $\mu$。
3. 不检查条件数。

---

## 3. 风险平价：按风险贡献分配

### 3.1 定义

风险贡献：

$$RC_i=\frac{w_i(\Sigma w)_i}{w^T\Sigma w}.$$

风险平价要求 $RC_i=1/N$。

### 3.2 近似

低相关时，$w_i\propto1/\sigma_i$ 是一个常用近似。

### 3.3 代码：逆波动 vs 风险贡献



In [4]:

w_rp = (1 / vol) / (1 / vol).sum()
port_var = w_rp @ cov @ w_rp
rc = w_rp * (cov @ w_rp) / port_var
print("逆波动权重", np.round(w_rp, 3))
print("风险贡献", np.round(rc, 3), "合计", round(rc.sum(), 3))


逆波动权重 [0.484 0.323 0.194]
风险贡献 [0.3  0.34 0.36] 合计 1.0



**读数**：逆波动权重给低波动资产更多；风险贡献接近但不完全相等（因为相关性）。严格风险平价要用数值优化。

**常见错误**：

1. 把逆波动当严格风险平价。
2. 忽略相关性。
3. 不做杠杆/约束。

---

## 4. Black–Litterman：均衡 + 观点

### 4.1 步骤

1. 市场均衡收益：$\pi=\delta\Sigma w_{\text{mkt}}$；
2. 观点：$P\mu=Q+\epsilon$，$\epsilon\sim N(0,\Omega)$；
3. 后验：

$$\mu_{\text{BL}}=\pi+\Sigma P^T(P\Sigma P^T+\Omega)^{-1}(Q-P\pi).$$

**公式拆解**
- **符号**：$\mu_{\text{BL}}$ 是后验预期收益向量，$\pi=\delta\Sigma w_{\text{mkt}}$ 是市场均衡收益，$P,Q$ 分别是观点矩阵与观点收益，$\Omega$ 是观点误差协方差（越小越自信）。
- **结构**：$\pi$ 是无观点时市场隐含的收益，$Q-P\pi$ 是观点与均衡的偏差，$(P\Sigma P^T+\Omega)^{-1}$ 按信心给偏差打折，$\Sigma P^T$ 再把偏差映射回资产空间。
- **代入**：取 $\pi=(6.4\%,1.6\%)$、绝对观点 $P=(1,0)$、$Q=8\%$、$\Omega=0.002$，调整量约为 $(0.0152,0.0015)$，后验收益 $\mu_{\text{BL}}\approx(7.92\%,1.75\%)$。
- **边界**：$\Omega\to0$ 时完全相信观点、$\Omega\to\infty$ 时退回均衡 $\pi$，$P$ 行不满秩或 $\Omega$ 非正定时矩阵不可逆。
- **用途**：把市场均衡与自己的观点按信心加权，得到比样本均值更稳的预期收益再进均值方差；它不是预测准确率，$\Omega$ 仍是主观设定。

4. 用 $\mu_{\text{BL}}$ 做均值方差。

### 4.2 代码：一个观点



In [5]:

delta = 2.5
w_mkt = np.array([0.5, 0.3, 0.2])
pi = delta * cov @ w_mkt
P = np.array([[0.0, 1.0, -1.0]])   # 资产 2 比资产 3 高 2%
Q = np.array([0.02])
Omega = np.array([[0.0004]])

mu_bl = pi + cov @ P.T @ np.linalg.inv(P @ cov @ P.T + Omega) @ (Q - P @ pi)
print("均衡收益", np.round(pi, 4))
print("BL 后验", np.round(mu_bl, 4))


均衡收益 [0.0185 0.03   0.0547]
BL 后验 [0.0143 0.0335 0.0139]



**读数**：观点把资产 2 的收益往上推、资产 3 往下推，但不是完全照搬观点，而是按不确定性加权。

**常见错误**：

1. $\Omega$ 设得太小，把观点当真理。
2. 不写市场均衡权重。
3. 把 BL 当“预测模型”，不检查输入。

---

## 5. 约束、换手与稳健

### 5.1 常见约束

- $0\le w_i\le0.4$；
- $\sum|w_i-w_i^0|\le0.3$（换手）；
- 行业/因子暴露；
- 整数手数/保证金。

### 5.2 稳健

- 重抽样/ bootstrap 看权重分布；
- 情景优化（多组 $\mu$）；
- 鲁棒优化（$\mu$ 的不确定集）。

### 5.3 代码：换手约束



In [6]:

w0 = np.array([1/3, 1/3, 1/3])
cons = [
    {"type": "eq", "fun": lambda w: w.sum() - 1},
    {"type": "ineq", "fun": lambda w: 0.3 - np.abs(w - w0).sum()},
]
res = minimize(lambda w: w @ cov @ w, w0, bounds=[(0, 0.5)] * 3, constraints=cons, method="SLSQP")
print("成功", res.success)
print("权重", np.round(res.x, 3), "换手", round(np.abs(res.x - w0).sum(), 3))


成功 True
权重 [0.483 0.333 0.183] 换手 0.3



**读数**：换手约束把权重拉回旧组合；约束越紧，越接近“不动”。**优化不是越“最优”越好，而是越“可执行”越好。**

**常见错误**：

1. 约束不可行还硬解。
2. 只加单品种上限，不加换手。
3. 不做权重分布/敏感性。

---

## 本章小结

### 一张表

| 方法 | 目标 | 优点 | 缺点 |
|---|---|---|---|
| 均值方差 | 给定风险最大收益 | 理论清晰 | 对 $\mu$ 敏感 |
| 风险平价 | 风险贡献相等 | 稳健/分散 | 收益低/需杠杆 |
| Black–Litterman | 均衡+观点 | 可解释 | 输入多 |
| 收缩 | 稳定协方差 | 降极端 | 有偏 |

### 八句话

1. 均值方差的最优权重常不可执行。
2. $\mu$ 的误差比 $\Sigma$ 更致命。
3. 收缩用偏差换方差。
4. 风险平价按风险贡献，不按资金。
5. 逆波动只是近似。
6. BL 把观点按不确定性加权。
7. 换手/上限/整数是硬约束。
8. 稳健比“理论最优”更重要。

---

## 本章习题

**1.** 最小方差组合的公式是什么？

**2.** 切点组合的公式是什么？

**3.** 为什么 $\mu$ 估错会让权重极端？

**4.** 风险平价的定义是什么？

**5.** 逆波动近似的假设是什么？

**6.** BL 后验公式里，$\Omega$ 控制什么？

**7.** 换手约束的作用是什么？

**8.** 为什么“最优权重”在实盘常不可用？

<details><summary>参考答案</summary>

**1.** $w=\Sigma^{-1}\mathbf1/(\mathbf1^T\Sigma^{-1}\mathbf1)$。

**2.** $w=\Sigma^{-1}\mu/(\mathbf1^T\Sigma^{-1}\mu)$。

**3.** $\Sigma^{-1}$ 放大误差，样本越短越极端。

**4.** 每个资产风险贡献相等。

**5.** 相关性低/忽略相关。

**6.** 观点的不确定性；越小越信观点。

**7.** 限制调仓幅度，降成本。

**8.** 卖空/整数/换手/容量/估计误差。

</details>

---

## 参考资料

- Markowitz, H. *Portfolio Selection*。
- Black, F. & Litterman, R. *Global Portfolio Optimization*。
- 第07、30、33章。
